# CI-MAI Evolutionary Computation Lab, Part 1: Genetic Algorithms

Lluís A. Belanche

In [ ]:
from datetime import datetime
print('Generated:', datetime.now().strftime('%d/%m/%Y %H:%M:%S'))

**Question of the day.** Two design decisions look like details: *how a solution is coded as a bitstring* and *how parents are selected*. How much do they change what a genetic algorithm (GA) achieves?

**How to work.** Work in pairs. Run the cells **in order**, one at a time. You do not need to write code: every cell runs as it is. Your work is to **think, predict and explain** in the coloured boxes:

- blue **Recall** boxes give the theory you need (from the GA lecture);
- yellow **Before running** boxes ask for a prediction *before* you run the next cell: write a shape and a reason, not a number;
- green **After running** boxes ask what you see and whether your prediction held.

Write your answers under *Your answers*. Lines marked *Try* are optional: change one value and rerun.

**Time.** This part takes about 50 minutes; Part 2 (evolution strategies) takes the rest of the session. Every cell runs in a few seconds, except the ones announced as slower.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
SEED = 2026      # numpy and matplotlib only, except Section 6 (deap)

# The canonical GA, in about 40 lines

<div style="background:#e8f1fb; border-left:6px solid #3b7dd8; padding:8px 12px; margin:6px 0;">

**Recall.** The canonical GA works on a **population** of $\mu$ bitstrings of length $n$:

1. initialise $\mu$ random bitstrings and evaluate their fitness;
2. repeat for a number of generations:
   - **select** $\mu$ parents according to fitness (roulette, SUS, tournament...);
   - **pair** them and, with probability $p_c$, apply **one-point crossover** (cut both parents at a random point and swap the tails);
   - **mutate** every bit of every child independently with probability $p_m$ (by default $p_m = 1/n$, one flip on average);
   - the children replace the parents, except that the best individual is kept (**elitism**);
3. return the best individual found.

The fitness is always **maximised**.

</div>

The next cell defines the whole GA. Read it with the Recall box at hand: it is the algorithm of the slide, line by line.

In [ ]:
# Selection: returns the indices of n parents, given the fitness vector
def select_parents(fit, n, method="tournament", k=2, rng=None):
    mu = len(fit)
    if method == "tournament":            # pick k at random, keep the best
        cand = rng.integers(0, mu, size=(n, k))
        return cand[np.arange(n), np.argmax(fit[cand], axis=1)]
    p = fit / fit.sum()                   # roulette and SUS need fit > 0
    if method == "roulette":              # n independent spins of the wheel
        return rng.choice(mu, size=n, p=p)
    if method == "sus":                   # one spin, n equally spaced pointers
        pointers = (rng.random() + np.arange(n)) / n
        return np.minimum(np.searchsorted(np.cumsum(p), pointers, side="right"), mu - 1)
    raise ValueError("unknown selection method")

# The canonical GA (maximises 'fitness', a function of a 0/1 vector)
def run_ga(fitness, nbits, pop=50, gens=100, pc=0.8, pm=None, selection="tournament", k=2, seed=1):
    rng = np.random.default_rng(seed)
    pm = 1 / nbits if pm is None else pm
    P = rng.integers(0, 2, size=(pop, nbits))                          # 1. initial population
    fit = np.array([fitness(x) for x in P], dtype=float)
    history = []
    for g in range(gens):                                              # 2. generations
        history.append((fit.max(), fit.mean()))
        elite = P[np.argmax(fit)].copy()
        parents = P[select_parents(fit, pop, selection, k, rng)]       # selection
        parents = parents[rng.permutation(pop)]                        # random pairing
        kids = parents.copy()
        for i in range(0, pop - 1, 2):                                 # one-point crossover
            if rng.random() < pc:
                cut = rng.integers(1, nbits)
                kids[i, cut:], kids[i + 1, cut:] = parents[i + 1, cut:], parents[i, cut:]
        flip = rng.random(kids.shape) < pm                             # bit-flip mutation
        kids[flip] = 1 - kids[flip]
        kids[0] = elite                                                # elitism
        P = kids
        fit = np.array([fitness(x) for x in P], dtype=float)
    history.append((fit.max(), fit.mean()))
    b = np.argmax(fit)
    return {"best": P[b], "fitness": fit[b], "history": np.array(history)}   # 3. result

<div style="background:#fff8dc; border-left:6px solid #d4a100; padding:8px 12px; margin:6px 0;">

**Before running.** Find in the code the lines that implement (a) selection, (b) crossover, (c) mutation and (d) elitism.
What would the GA become if we set $p_c = 0$? And if we also removed selection (parents taken at random)?

*Your answers:*

</div>

# Coding a real number

<div style="background:#e8f1fb; border-left:6px solid #3b7dd8; padding:8px 12px; margin:6px 0;">

**Recall.** To represent $x \in [a, b]$ with **precision** $\varepsilon$ (that is, $|x - \mathrm{dec}(\mathrm{enc}(x))| < \varepsilon$) we use $k$ bits:

- $2^k$ equally spaced points, i.e. $2^k - 1$ segments of length $(b-a)/(2^k-1)$, labelled $0, \dots, 2^k - 1$;
- the error is at most half a segment, $\dfrac{b-a}{2(2^k-1)} < \varepsilon$, which gives
$$k = \left\lfloor \log_2\!\Big(\frac{b-a}{2\varepsilon} + 1\Big) \right\rfloor + 1;$$
- decoding: if the bits $b_1 \dots b_k$ represent the integer $z$, then $\mathrm{dec}(z) = a + z\,\dfrac{b-a}{2^k-1}$.

</div>

We maximise $f(x) = (x^2 + x)\cos x$ on $[-10, 10]$. Its maximum is $f(x^\ast) \approx 47.7056$ at $x^\ast \approx 6.56054$ (found numerically on a fine grid).

<div style="background:#fff8dc; border-left:6px solid #d4a100; padding:8px 12px; margin:6px 0;">

**Before running.** How many bits $k$ do we need for a precision $\varepsilon = 10^{-3}$? Estimate it by hand from the formula before running the cell.
If we asked for $\varepsilon = 10^{-6}$ instead, would $k$ grow a lot or a little? Why?

*Your answers:*

</div>

In [ ]:
a, b, eps = -10, 10, 1e-3
k = int(np.floor(np.log2((b - a) / (2 * eps) + 1))) + 1
print("bits needed:", k, "   half segment:", (b - a) / (2 * (2**k - 1)))

In [ ]:
def decode(bits, a, b):                                # bits -> integer z -> a + z (b-a)/(2^k-1)
    k = len(bits)
    return a + int("".join(str(int(v)) for v in bits), 2) * (b - a) / (2**k - 1)

f = lambda x: (x**2 + x) * np.cos(x)
res = run_ga(lambda bits: f(decode(bits, a, b)), nbits=k, pop=50, gens=100, seed=SEED)
x_best = decode(res["best"], a, b)
print("best string:", res["best"])
print("x =", x_best, "  f(x) =", res["fitness"], "  |x - x*| =", abs(x_best - 6.56054))
xs = np.linspace(-10, 10, 1000)
plt.plot(xs, f(xs)); plt.plot(x_best, res["fitness"], "ro"); plt.xlabel("x"); plt.ylabel("f(x)"); plt.show()

<div style="background:#eaf7ea; border-left:6px solid #3a9a3a; padding:8px 12px; margin:6px 0;">

**After running.** Did the GA find the global maximum (there is a competing local maximum near $x \approx -6.60$, with $f \approx 35.1$)? Is the error $|x - x^\ast|$ below $\varepsilon$?
Could a GA with these 14 bits ever return $x^\ast$ **exactly**?

*Your answers:*

</div>

# Binary or Gray code? One knob: the code

<div style="background:#e8f1fb; border-left:6px solid #3b7dd8; padding:8px 12px; margin:6px 0;">

**Recall.**

- In **plain binary**, numerically adjacent values can be far apart in Hamming distance (**Hamming cliffs**): $7 = 0111$, $8 = 1000$. To move from 7 to 8, mutation must flip 4 bits at once.
- In a **Gray code**, consecutive values differ in exactly one bit. From Gray $g_1 \dots g_k$ to binary: $b_1 = g_1$, $b_i = b_{i-1} \oplus g_i$, i.e. $b_i = (g_1 + \dots + g_i) \bmod 2$.
- Gray removes the cliffs, but not the long jumps (flipping the first bit still moves the value far), and it changes the epistasis of the problem.

</div>

We minimise the **Rastrigin function** in $D = 5$ dimensions, $f(\mathbf x) = 10D + \sum_{i=1}^D \big(x_i^2 - 10\cos(2\pi x_i)\big)$ on $[-5.12, 5.12]^D$: a regular grid of local minima, with the global minimum $f(\mathbf 0) = 0$ (on this 10-bit grid $\mathbf 0$ is not representable, and the best reachable value is $f \approx 0.025$).
Each variable uses $k = 10$ bits, so a string has 50 bits; the GA maximises $-f$. **Everything is the same in the two runs except the code.** We run the GA 20 times with each code (different seeds) and compare the final values.

<div style="background:#fff8dc; border-left:6px solid #d4a100; padding:8px 12px; margin:6px 0;">

**Before running.** Which code do you expect to reach lower values of $f$, and why? Think of what a single bit flip does near a local minimum in each code.

*Your answers:*

</div>

In [ ]:
# decode D variables of k bits each; with gray=True, the bits are read as a Gray code
def decode_vec(bits, D, a, b, gray=False):
    B = np.asarray(bits).reshape(D, -1); k = B.shape[1]
    if gray:
        B = np.cumsum(B, axis=1) % 2                    # Gray -> binary: b_i = (g_1+...+g_i) mod 2
    return a + (B @ 2**np.arange(k - 1, -1, -1)) * (b - a) / (2**k - 1)

rastrigin = lambda x: 10 * len(x) + np.sum(x**2 - 10 * np.cos(2 * np.pi * x))
D, kb = 5, 10
final = {}
for name, g in [("binary", False), ("gray", True)]:
    final[name] = np.array([-run_ga(lambda bits: -rastrigin(decode_vec(bits, D, -5.12, 5.12, g)),
                                    nbits=D * kb, pop=60, gens=150, seed=s)["fitness"] for s in range(1, 21)])
    print(f"{name:7s} median {np.median(final[name]):.3f}   runs with f < 0.5: {(final[name] < 0.5).sum()}")
plt.boxplot([final["binary"], final["gray"]], tick_labels=["binary", "gray"]); plt.ylabel("final f (lower is better)"); plt.show()

*This cell is the slowest of Part 1 (about 20 s; up to a minute on a slow laptop).*

<div style="background:#eaf7ea; border-left:6px solid #3a9a3a; padding:8px 12px; margin:6px 0;">

**After running.** Which code wins? Is the difference large compared with the spread between runs of the same code?
Would you trust a conclusion drawn from a single run of each?

*Your answers:*

</div>

*Try*: set `gens` to 300 in both. Does the gap close?

# Selection: roulette, SUS and tournament

<div style="background:#e8f1fb; border-left:6px solid #3b7dd8; padding:8px 12px; margin:6px 0;">

**Recall.** Running example of the lecture: $\mu = 6$ individuals A, ..., F with fitness 9, 6, 4, 3, 2, 1 (total 25).

- **Roulette**: $p_i = f_i / \sum_j f_j$; spin $\mu$ times, independently. A has $p_A = 0.36$ and expects $\mu p_A = 2.16$ copies, but the number of copies is $\mathrm{Bin}(6, 0.36)$.
- **SUS** (stochastic universal sampling): the same wheel, but **one** spin with $\mu$ equally spaced pointers. Each individual gets $\lfloor \mu p_i \rfloor$ or $\lceil \mu p_i \rceil$ copies: the same expectation with minimal spread.
- **Tournament** ($k = 2$): pick 2 at random, keep the better one. Only the **order** of the fitness values matters.

</div>

<div style="background:#fff8dc; border-left:6px solid #d4a100; padding:8px 12px; margin:6px 0;">

**Before running.** Over many generations, how often will A get **no** copy at all with the roulette? And with SUS? (A formula is enough.)

*Your answers:*

</div>

In [ ]:
fitAF = np.array([9, 6, 4, 3, 2, 1.])
rng = np.random.default_rng(SEED)
for m in ["roulette", "sus"]:
    cA = np.array([np.sum(select_parents(fitAF, 6, m, rng=rng) == 0) for _ in range(10000)])
    print(f"{m:9s} copies of A, frequency of 0..6:", np.round(np.bincount(cA, minlength=7) / 10000, 3))

<div style="background:#eaf7ea; border-left:6px solid #3a9a3a; padding:8px 12px; margin:6px 0;">

**After running.** Compare the two rows. Which scheme is noisier? Why does that noise matter in a GA that runs for many generations?

*Your answers:*

</div>

Now a whole GA on **OneMax** (the COP of the lecture): maximise the number of ones in a string of $n = 64$ bits. We compare roulette, SUS and tournament, each on the fitness $f$ and on $f + 100$. Adding a constant does not change the problem: same best string, same order of all strings. Each curve is the mean of 10 runs.

<div style="background:#fff8dc; border-left:6px solid #d4a100; padding:8px 12px; margin:6px 0;">

**Before running.** Which curves will be **identical** for $f$ and $f + 100$? Which scheme will be hurt most by adding 100, and why? (Recall the lecture: what happens to the roulette probabilities if we add a constant?)

*Your answers:*

</div>

In [ ]:
configs = [("roulette", 0), ("roulette", 100), ("sus", 0), ("sus", 100), ("tournament", 0), ("tournament", 100)]
colors = {"roulette": "red", "sus": "darkgreen", "tournament": "blue"}
for m, C in configs:
    runs = [run_ga(lambda bits: bits.sum() + C, nbits=64, pop=40, gens=60, selection=m, seed=s)["history"][:, 0] - C
            for s in range(1, 11)]
    curve = np.mean(runs, axis=0)
    label = f"{m} {'f' if C == 0 else 'f+100'}"
    plt.plot(curve, color=colors[m], linestyle="-" if C == 0 else "--", label=label)
    print(f"{label:16s} final mean best: {curve[-1]:.1f}")
plt.xlabel("generation"); plt.ylabel("best number of ones (mean of 10 runs)"); plt.legend(); plt.show()

<div style="background:#eaf7ea; border-left:6px solid #3a9a3a; padding:8px 12px; margin:6px 0;">

**After running.** Did your prediction hold? Rank the six curves. Why is tournament selection the usual default today?

*Your answers:*

</div>

# A real problem: the 0--1 knapsack

<div style="background:#e8f1fb; border-left:6px solid #3b7dd8; padding:8px 12px; margin:6px 0;">

**Recall.** $n$ items with weights $w_i$ and values $v_i$, and a capacity $W$. Choose $x \in \{0,1\}^n$ to maximise $\sum_i v_i x_i$ subject to $\sum_i w_i x_i \le W$. The bitstring is the natural representation: bit $i$ says whether item $i$ goes in.
The decision version of the problem is NP-complete, but for integer weights and small $W$ it is solved **exactly** by dynamic programming in $O(nW)$ time, so here we can check the GA against the true optimum.
Infeasible strings get a **penalty**: their fitness is $W - \sum_i w_i x_i < 0$, below every feasible string. (The empty knapsack is always feasible, so a solution always exists.)

</div>

In [ ]:
import io, csv
data = '''item,weight,value
map,11,13
compass,3,17
raincoat,1,14
first-aid kit,8,9
fire starter,5,10
knife,6,7
mineral water,1,4
water purifier,5,6
gloves,8,3
sleeping bag,5,5
tent,10,18
portable stove,10,11
canned food,9,2
chocolate,11,9
cordage,9,6
fishing kit,3,14
blanket,11,17
cooking pots,8,1
water filter,2,10
hatchet,4,9
saw,2,20
paracord,12,8
poncho,9,4
sunblock,12,4
phone charger,4,10
shovel,8,12
utensils,8,7
flashlight,10,9
whistle,2,14
radio,7,6'''
rows = list(csv.DictReader(io.StringIO(data)))
names = [r["item"] for r in rows]
w = np.array([int(r["weight"]) for r in rows]); v = np.array([int(r["value"]) for r in rows])
W = 50
print(len(names), "items, total weight", w.sum(), ", capacity W =", W)
x_random = np.random.default_rng(SEED).integers(0, 2, len(w))
print("a random string: weight", x_random @ w, "-> infeasible?", x_random @ w > W)

The exact optimum, by dynamic programming (you do not need to read this cell):

In [ ]:
def knapsack_dp(w, v, W):
    best = np.zeros(W + 1, int); keep = np.zeros((len(w), W + 1), bool)
    for i in range(len(w)):
        for cap in range(W, w[i] - 1, -1):
            if best[cap - w[i]] + v[i] > best[cap]:
                best[cap] = best[cap - w[i]] + v[i]; keep[i, cap] = True
    x = np.zeros(len(w), int); cap = W
    for i in range(len(w) - 1, -1, -1):
        if keep[i, cap]: x[i] = 1; cap -= w[i]
    return best[W], x
opt_value, opt_x = knapsack_dp(w, v, W)
print("optimal value:", opt_value, "  weight:", opt_x @ w)

<div style="background:#fff8dc; border-left:6px solid #d4a100; padding:8px 12px; margin:6px 0;">

**Before running.** We run the GA (tournament, $\mu = 50$, 100 generations: about 5000 evaluations, against $2^{30} \approx 10^9$ possible strings) 20 times with different seeds.
In how many of the 20 runs do you expect it to reach the optimum? And how far below the optimum do you expect the others to be? Write a guess.

*Your answers:*

</div>

In [ ]:
fit_knap = lambda x: x @ v if x @ w <= W else W - x @ w
ks = np.array([run_ga(fit_knap, nbits=len(w), pop=50, gens=100, seed=s)["fitness"] for s in range(1, 21)])
vals, counts = np.unique(ks, return_counts=True)
print(dict(zip(vals.astype(int).tolist(), counts.tolist())))
print("runs at the optimum:", (ks == opt_value).sum(), "of 20;  worst run:", ks.min())

<div style="background:#eaf7ea; border-left:6px solid #3a9a3a; padding:8px 12px; margin:6px 0;">

**After running.** Compare with your guess. Is the GA useful here, even when it misses the optimum? What would you change first to improve it (population, generations, selection, a repair step...)?

*Your answers:*

</div>

# From the toy GA to a package (for your assignment)

*If you are short of time, do this section at home: you will need it for your assignment.*

Our 40-line GA is good for understanding, but for real work you will use a package that lets you plug in your own operators and stopping rules: in R the package **GA** (Scrucca 2013), in Python the package **DEAP** (Fortin et al. 2012). Here we solve the same knapsack with three typical ingredients:

1. a **problem-specific operator**: a mutation that flips one bit and then **repairs** the string, removing random items until it fits in the knapsack, so that every individual is feasible;
2. a **stopping rule**: stop when the best value has not improved for 50 generations (at most 500 generations);
3. a **convergence curve** of the best value.

In [ ]:
# In Google Colab, or if 'deap' is not installed, uncomment the next line:
# %pip install deap
import random
from deap import base, creator, tools, algorithms

creator.create("FitnessMax", base.Fitness, weights=(1.0,))      # we maximise
creator.create("Individual", list, fitness=creator.FitnessMax)

def repair_mutation(ind):                            # the signature that DEAP expects
    j = random.randrange(len(ind)); ind[j] = 1 - ind[j]          # flip one random bit
    while np.dot(ind, w) > W:                        # repair: drop random items until it fits
        inside = [i for i, b in enumerate(ind) if b == 1]; ind[random.choice(inside)] = 0
    return ind,

tb = base.Toolbox()
tb.register("bit", random.randint, 0, 1)
tb.register("individual", tools.initRepeat, creator.Individual, tb.bit, n=len(w))
tb.register("population", tools.initRepeat, list, tb.individual)
tb.register("evaluate", lambda x: (float(fit_knap(np.array(x))),))
tb.register("mate", tools.cxOnePoint)
tb.register("mutate", repair_mutation)
tb.register("select", tools.selTournament, tournsize=2)

random.seed(SEED)
pop = tb.population(n=50)
for ind in pop: ind.fitness.values = tb.evaluate(ind)
hof = tools.HallOfFame(1); hof.update(pop)
best, stall, gen = [hof[0].fitness.values[0]], 0, 0
while gen < 500 and stall < 50:                      # stopping rule
    gen += 1
    off = algorithms.varAnd(tb.select(pop, len(pop)), tb, cxpb=0.8, mutpb=0.2)
    for ind in off:
        if not ind.fitness.valid: ind.fitness.values = tb.evaluate(ind)
    off[0] = tb.clone(hof[0])                        # elitism
    pop = off
    old = hof[0].fitness.values[0]; hof.update(pop)
    stall = stall + 1 if hof[0].fitness.values[0] <= old else 0
    best.append(hof[0].fitness.values[0])
print("best value:", hof[0].fitness.values[0], "  (optimum", opt_value, ")   stopped after", gen, "generations")
plt.plot(best); plt.xlabel("generation"); plt.ylabel("best value"); plt.show()   # convergence curve

<div style="background:#eaf7ea; border-left:6px solid #3a9a3a; padding:8px 12px; margin:6px 0;">

**After running.** Where in the code are the three ingredients? With the repair operator, can an infeasible string ever survive? Is the penalty in `fit_knap` still needed?

*Your answers:*

</div>

# Wrap-up

**Answer to the question of the day.** Write two sentences: what did the code change on Rastrigin, and what did the selection scheme change on OneMax?

**Beyond this lab.** For your assignment use a package (Section 6): the R package `GA` (Scrucca 2013, 2017) or the Python package `DEAP`. Both accept your own selection, crossover and mutation functions and stopping rules, and both provide operators for **permutations** (order and partially mapped crossover, swap and insertion mutation). R `GA` also offers local search hybrids (`optim = TRUE`), island models (`gaisl()`) and parallel evaluation; `DEAP` also offers genetic programming (trees), multi-objective selection and any scheme you can write as a loop (e.g. coevolution).

**Homework (AI audit).** Ask an AI assistant, without context: *"Is SUS better than roulette-wheel selection? Why?"* Judge its answer: correct, correct for the wrong reason, or wrong. Use what you saw in Section 4.

**References.** F.-A. Fortin et al., DEAP: evolutionary algorithms made easy, *J. Machine Learning Research* 13:2171--2175, 2012. D. E. Goldberg, *Genetic Algorithms in Search, Optimization and Machine Learning*, Addison-Wesley, 1989. J. E. Baker, Reducing bias and inefficiency in the selection algorithm, *ICGA 2*, 1987 (SUS). L. Scrucca, GA: a package for genetic algorithms in R, *J. Statistical Software* 53(4), 2013; On some extensions to GA package, *The R Journal* 9(1), 2017.

In [ ]:
import sys, platform, matplotlib
import deap
print(sys.version); print(platform.platform()); print("numpy", np.__version__, " matplotlib", matplotlib.__version__, " deap", deap.__version__)